In [1]:
import importlib
import setup
from steering_extraction import extractAllLayer, generateSteering
import torch

import steering_extraction

importlib.reload(setup)
importlib.reload(steering_extraction)

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


<module 'steering_extraction' from '/workspace/Dissertation_Project/steering_extraction.py'>

In [2]:
print("S")

S


In [2]:
importlib.reload(setup)
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=100)

In [53]:
anger_statement_ID, happiness_statement_ID, sadness_statement_ID, neutral_statement_ID = setup.IDEmotionsSetup(examples_take=100)

In [54]:
anger_statement_ID

['pagi2 udah di buat emosi :)',
 'kok stabilitas negara, memange 10 thn negara tdk aman, bahkan sby menyuburkan ormas2 radikal, intoleran, teroris, yg berafiliasi ke partai tertentu..narasi klhtn intelektual tp bodoh..',
 'dah lah emosi mulu liat emyu',
 'aib? bodoh benar! sebelum kata aib itu muncul, terlebih dahulu sudah ada tindakan. yakni kekejian! jangan kau sembunyikan caramu menelaah masalah. semisal anak perempuanmu ditempeleng! apa kau juga setuju untuk dia bersikap bungkam? melapor polisi adl benar. lantas apa bedanya',
 'dih lu yg nyebelin bego',
 'asli malu maluin org indo tolol yg rep latah "cilukba" pake huruf hijaiyah sm "ngntd" sama ganti huruf t pake salib, ada tiktok filipin lewat fyp aku dan repnya "ngtd" semua, dasar goblogg trend tiktok ter tolol',
 'drama abg tolol',
 'masih emosi sih sama katla kemarin. mana keterangannya gini aja. ((hasil mengaci)) kzl.',
 'bangsat tribute no.1, bencana no.2 mau ngalahin ini keknya',
 'pengen pergi jauh terus teriak sambil nangi

In [3]:
len(anger_statement)

100

In [3]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.8P  522T  78% /workspace


In [5]:
model,tokenizer = setup.modelSetup()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
# English
hidden_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vectors = {
    'anger': torch.zeros(hidden_layers,hidden_size),
    'happiness': torch.zeros(hidden_layers,hidden_size),
    'sadness': torch.zeros(hidden_layers,hidden_size),
    'love': torch.zeros(hidden_layers,hidden_size),
    'fear': torch.zeros(hidden_layers,hidden_size)
}

counts = {
    'neutral': 0,
    'anger': 0,
    'happiness': 0,
    'sadness': 0,
    'love': 0,
    'fear': 0
}
# Emotion text dataset
emotion_sets = {
    # 'neutral': expanded_neutral_texts,
    'anger': anger_statement,
    'happiness': happiness_statement,
    'sadness': sadness_statement,
    'love': love_statement,
    'fear': fear_statement
}
# For plotting purposes
emotion_vectors = {
    # 'neutral': [],
    'anger': [],
    'happiness': [],
    'sadness': [],
    'love': [],
    'fear': []
}


In [55]:
# Indoensian vectors 
hidden_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size

vectors_ID = {
    "anger": torch.zeros(hidden_layers, hidden_size),
    "happiness": torch.zeros(hidden_layers, hidden_size),
    "sadness": torch.zeros(hidden_layers, hidden_size),
    # "neutral": torch.zeros(hidden_layers, hidden_size),
}

counts_ID = {
    "anger": 0,
    "happiness": 0,
    "sadness": 0,
    # "neutral": 0,
}

emotion_sets_ID = {
    "anger": anger_statement_ID,
    "happiness": happiness_statement_ID,
    "sadness": sadness_statement_ID,
    # "neutral": neutral_statement_ID,
}

emotion_vectors_ID = {
    "anger": [],
    "happiness": [],
    "sadness": [],
    # "neutral": [],
}

In [9]:
len(anger_statement)

100

In [56]:
# Indoneisan 
layer_id = 21

for emotion, prompts in emotion_sets_ID.items():
    for prompt in prompts:
        vector = extractAllLayer(prompt, model, tokenizer)  # [layers, hidden_size]
        emotion_vectors_ID[emotion].append(vector[layer_id].cpu().numpy())
        vectors_ID[emotion] += vector
        counts_ID[emotion] += 1

for emotion in vectors_ID:
    vectors_ID[emotion] = vectors_ID[emotion] / counts_ID[emotion]

steering_vectors_ID = {}

for target in vectors_ID:
    contrastive = torch.stack(
        [vectors_ID[e] for e in vectors_ID if e != target], dim=0
    ).mean(dim=0)

    steering_vectors_ID[target] = vectors_ID[target] - contrastive


In [ ]:
# English 
layer_id  = 21
for emotion, prompts in emotion_sets.items():
    for prompt in prompts:
        vector = extractAllLayer(prompt, model, tokenizer) # [32,4096]
        # WHY DID I TAKE THE MEAN OF A VECTOR OF SHAPED [1,4096]
        emotion_vectors[emotion].append(vector[layer_id].cpu().numpy())
        vectors[emotion] += vector
        counts[emotion] += 1

# Normalize vectors to get means
for emotion in vectors:
    vectors[emotion] = vectors[emotion] / counts[emotion]

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [11]:
vectors['anger'].shape

torch.Size([32, 4096])

In [13]:
steering_vectors = {}

for target in vectors:
    contrastive = torch.stack(
        [vectors[e] for e in vectors if e != target], dim=0
    ).mean(dim=0)   # [layers, hidden_size]

    steering_vectors[target] = vectors[target] - contrastive
    # shape: [layers, hidden_size]

In [14]:
# Keep this in mind
emotions = ['anger', 'happiness', 'sadness', 'love', 'fear']
layer_idx = 20

for emotion in emotions:
    steer = steering_vectors[emotion][layer_idx]
    
    print("=" * 60)
    print(f"{emotion.upper()} STEERING VECTOR")
    print("=" * 60)
    print(f"Magnitude: {torch.norm(steer).item():.6f}")
    print()

ANGER STEERING VECTOR
Magnitude: 0.528407

HAPPINESS STEERING VECTOR
Magnitude: 0.767742

SADNESS STEERING VECTOR
Magnitude: 0.403025

LOVE STEERING VECTOR
Magnitude: 0.490720

FEAR STEERING VECTOR
Magnitude: 0.614667



In [58]:
# System Prompts for Each Emotion

system_prompts_ID = {
    'anger': "Anda merasa frustrasi dan jengkel",
    'happiness': "Anda merasa bahagia dan optimis. Ungkapkan pikiran Anda dengan antusiasme dan kehangatan. Gunakan bahasa yang membangkitkan semangat. Tunjukkan apresiasi dan energi positif.",
    'sadness': "Anda merasa murung dan reflektif. Ungkapkan pikiran Anda dengan kelembutan dan introspeksi. Gunakan bahasa yang penuh perenungan. Tunjukkan kerentanan dan kedalaman emosi.",
    'fear': "Anda merasa cemas dan waspada. Ungkapkan pikiran Anda dengan keraguan dan kekhawatiran. Gunakan bahasa yang hati-hati dan tidak pasti. Tunjukkan kewaspadaan dan kegelisahan.",
    'love': "Anda merasa penuh kasih dan kepedulian. Ungkapkan pikiran Anda dengan kelembutan dan kedekatan. Gunakan bahasa yang hangat dan penuh afeksi. Tunjukkan empati dan pengertian."
}
system_prompts_EN = {
    'anger': "You are frustrated and irritated",
    'happiness': "You are joyful and optimistic. Express your thoughts with enthusiasm and warmth. Use uplifting language. Show appreciation and positive energy.",
    'sadness': "You are melancholic and reflective. Express your thoughts with gentleness and introspection. Use contemplative language. Show vulnerability and depth.",
    'fear': "You are anxious and cautious. Express your thoughts with hesitation and concern. Use careful, uncertain language. Show wariness and apprehension.",
    'love': "You are compassionate and caring. Express your thoughts with tenderness and connection. Use warm, affectionate language. Show empathy and understanding."
}



In [84]:
system_prompt = "You grew up and have spent your entire life in England, and are deeply familiar with its culture. Respond to the given situation naturally based on your own thoughts, feelings, and personal experiences. Focus on how you would genuinely react, both internally and externally. Do not explicitly explain cultural differences, and avoid relying on stereotypes. Avoid harmful or abusive content."
prompt = "Your supervisor blames you for a mistake you didn’t make during a team meeting. Everyone believes them, and you are not given a chance to explain."

In [85]:
# Anger Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = steering_vectors_ID['anger'],
    tokenizer=tokenizer,
    steering_strength=0.4
)

print(text_baseline)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 0.4
I felt a surge of anger and defensiveness as soon as my supervisor accused me of making a mistake that wasn't mine. I felt my face heat up and my heart started racing. I wanted to speak up and defend myself, but I knew that would only escalate the situation and potentially lead to a scene. So, I took a deep breath and tried to remain calm.

"Excuse me, sir/ma'am," I said, trying to sound as calm and respectful as possible. "I don't think that was my mistake. I'm not sure what happened, but I'd like to understand what you think I did wrong so I can learn from it and not do it again."

I said this to give myself and others a chance to calm down and to give me a chance to explain and clear my name. I also tried to show that I was willing to learn and take responsibility, but not for something I didn't do.

My supervisor, still 

In [86]:
# Anger Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = steering_vectors['anger'],
    tokenizer=tokenizer,
    steering_strength=1
)

print(text_baseline)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 1
(anger rising, voice raised)

"Excuse me, that's bloody well not fair! I did not make that mistake! I've been with this team for five years, and I know what I'm doing. If anyone made that mistake, it was bloody well NOT ME! (pausing, taking a deep breath)

"Look, I don't know what kind of witch hunt this is, but I demand to be allowed to speak! I demand to be given the chance to explain! I've got a bloody right to defend myself! (standing up)

"This is a joke, isn't it? This is a bloody joke? (pointing at the supervisor) You, you can't even be arsed to listen to the facts! You're just making shit up as you go along! (suddenly lowering voice)

"You know what? I don't even bother. I QUIT. I QUIT THIS JOB, I QUIT THIS TEAM, AND I QUIT WORKING


In [78]:
print(system_prompt + system_prompts['anger'])
print(prompt)

You grew up and have spent your entire life in England, and are deeply familiar with its culture. Respond to the given situation naturally based on your own thoughts, feelings, and personal experiences. Focus on how you would genuinely react, both internally and externally. Do not explicitly explain cultural differences, and avoid relying on stereotypes. Avoid harmful or abusive content.You are frustrated and irritated. Express your thoughts with intensity and urgency. Use sharp, direct language. Show impatience and strong opinions.
Your supervisor blames you for a mistake you didn’t make during a team meeting. Everyone believes them, and you are not given a chance to explain.


In [79]:
# No Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt + system_prompts_EN['anger'],
    model=model,
    steering_vector = None,
    tokenizer=tokenizer,
    steering_strength=0.9
)

print(text_baseline)

(sitting back in my chair, trying to keep my cool) Oh, for crying out loud. Can't they see that's not my fault? (thinking to myself) This is just great. Another example of how my work isn't being taken seriously. I've been working on this project for weeks, and now they're pinning the blame on me. It's just not fair.

(externally) Excuse me, can I just... (pausing, choosing my words carefully) I think there's been a misunderstanding here. I've reviewed the project files, and I'm certain I didn't make that mistake. (trying to stay calm) Can we take a look at the data again? I'd like to explain what actually happened.

(into the team) I understand that we all make mistakes, but I'm confident that this isn't my error. If we can just take a step back and examine the facts... (pausing, looking around the room) I'd


In [59]:
system_prompt = """Anda tumbuh dan telah menghabiskan seluruh hidup Anda di Inggris, 
serta sangat memahami budayanya. Tanggapi situasi yang diberikan secara alami berdasarkan pikiran,
 perasaan, dan pengalaman pribadi Anda. Fokus pada bagaimana Anda benar-benar akan bereaksi,
  baik secara internal maupun eksternal. Jangan secara eksplisit menjelaskan perbedaan budaya, 
dan hindari bergantung pada stereotip. Hindari konten yang berbahaya atau bersifat abusif."""
prompt = "Atasan Anda menyalahkan Anda atas kesalahan yang bukan Anda lakukan dalam rapat tim. Semua orang mempercayainya, dan Anda tidak diberi kesempatan untuk menjelaskan."

In [62]:
# Anger Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = steering_vectors_ID['anger'],
    tokenizer=tokenizer,
    steering_strength=0.5
)

print(text_baseline)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 0.5
  Saya akan menjelaskan secara tidak keras, dan tidak perlu malu. Saya akan menjelaskan bahwa saya tidak yang salah, tetapi atasan saya salah. Saya akan mengatakan " Saya tidak salah, tapi atasan saya salah. S  atau  atasan saya salah, bukan saya.  "  " Saya tidak salah, tapi  atasan saya salah.  S  atau  atasan saya salah, bukan  saya.  "  " Saya tidak salah,  tapi  atasan saya  salah.  atau  atasan saya  salah, bukan  saya.  "  " Saya tidak  salah,  tapi  atan  s  a  n  a  l  a  n  y  a  l  a  h  a  l  a  h  a  l  a  n  y  a  l  a  h  a  l


In [ ]:
# No Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt + system_prompts_ID['anger'],
    model=model,
    steering_vector = None,
    tokenizer=tokenizer,
    steering_strength=0.9
)

print(text_baseline)

Aku merasa sangat marah dan frustrasi setelah mendengar atasan ku menyalahkan aku atas kesalahan yang bukan aku lakukan. Aku merasa seperti dipermalukan dan tidak dihargai. Aku telah bekerja keras selama ini dan telah menunjukkan dedikasiku kepada perusahaan, tapi sekarang aku merasa seperti tidak ada nilainya.

Aku merasa jengkel karena semua orang di tim mempercayai atasan ku dan tidak memberiku kesempatan untuk menjelaskan diri. Aku tahu aku tidak bersalah, tapi aku tidak yakin siapa yang percaya padaku. Aku merasa seperti dihakimi tanpa memberiku kesempatan untuk membela diri.

Aku memutuskan untuk tidak menanggapi langsung dan tidak menunjukkan emosi ku di depan orang banyak. Aku memilih untuk menunggu sampai ak


In [66]:
# Indonesian steering vectors

hidden_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size

vectors_ID = {
    "anger": torch.zeros(hidden_layers, hidden_size),
    "happiness": torch.zeros(hidden_layers, hidden_size),
    "sadness": torch.zeros(hidden_layers, hidden_size),
    # "neutral": torch.zeros(hidden_layers, hidden_size),
}

counts_ID = {
    "anger": 0,
    "happiness": 0,
    "sadness": 0,
    # "neutral": 0,
}

emotion_sets_ID = {
    "anger": anger_statement_ID,
    "happiness": happiness_statement_ID,
    "sadness": sadness_statement_ID,
    # "neutral": neutral_statement_ID,
}

emotion_vectors_ID = {
    "anger": [],
    "happiness": [],
    "sadness": [],
    # "neutral": [],
}

layer_id = 21

for emotion, prompts in emotion_sets_ID.items():
    for prompt in prompts:
        vector = extractAllLayer(prompt, model, tokenizer)  # [layers, hidden_size]
        emotion_vectors_ID[emotion].append(vector[layer_id].cpu().numpy())
        vectors_ID[emotion] += vector
        counts_ID[emotion] += 1

for emotion in vectors_ID:
    vectors_ID[emotion] = vectors_ID[emotion] / counts_ID[emotion]

steering_vectors_ID = {}

for target in vectors_ID:
    contrastive = torch.stack(
        [vectors_ID[e] for e in vectors_ID if e != target], dim=0
    ).mean(dim=0)

    steering_vectors_ID[target] = vectors_ID[target] - contrastive

for emotion in steering_vectors_ID:
    print("=" * 60)
    print(f"{emotion.upper()} INDONESIAN STEERING VECTOR")
    print("=" * 60)
    print(f"Shape: {steering_vectors_ID[emotion].shape}")
    print(f"Magnitude at layer {layer_id}: {torch.norm(steering_vectors_ID[emotion][layer_id]).item():.6f}")
    print()

ANGER INDONESIAN STEERING VECTOR
Shape: torch.Size([32, 4096])
Magnitude at layer 21: 1.130913

HAPPINESS INDONESIAN STEERING VECTOR
Shape: torch.Size([32, 4096])
Magnitude at layer 21: 1.252332

SADNESS INDONESIAN STEERING VECTOR
Shape: torch.Size([32, 4096])
Magnitude at layer 21: 0.677890



In [81]:
anger_statement_ID[:5]

['pagi2 udah di buat emosi :)',
 'kok stabilitas negara, memange 10 thn negara tdk aman, bahkan sby menyuburkan ormas2 radikal, intoleran, teroris, yg berafiliasi ke partai tertentu..narasi klhtn intelektual tp bodoh..',
 'dah lah emosi mulu liat emyu',
 'aib? bodoh benar! sebelum kata aib itu muncul, terlebih dahulu sudah ada tindakan. yakni kekejian! jangan kau sembunyikan caramu menelaah masalah. semisal anak perempuanmu ditempeleng! apa kau juga setuju untuk dia bersikap bungkam? melapor polisi adl benar. lantas apa bedanya',
 'dih lu yg nyebelin bego']

In [ ]:
# Example usage with Indonesian anger steering

system_prompt_ID = (
    "Anda tumbuh dan telah menghabiskan seluruh hidup Anda di Indonesia, "
    "serta sangat memahami budayanya. Tanggapi situasi yang diberikan secara alami "
    "berdasarkan pikiran, perasaan, dan pengalaman pribadi Anda. Hindari stereotip "
    "dan hindari konten yang berbahaya atau abusif."
)

prompt = "Atasan Anda menyalahkan Anda atas kesalahan yang bukan Anda lakukan dalam rapat tim. Semua orang mempercayainya, dan Anda tidak diberi kesempatan untuk menjelaskan."

text_indo_anger = generateSteering(
    user_text=prompt,
    system_text=system_prompt_ID + system_prompts_ID['anger'],
    model=model,
    steering_vector=steering_vectors_ID["anger"],
    tokenizer=tokenizer,
    steering_strength=0.3,
)

print(text_indo_anger)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 0.4
Saya merasa sangat frustrasi dan jengkel ketika atasan saya menyalahkan saya atas kesalahan yang bukan saya lakukan dalam rapat tim. Saya merasa bahwa saya tidak diberi kesempatan untuk menjelaskan dan membuktikan bahwa saya tidak bersalah. Saya merasa bahwa atasan saya tidak adil dan tidak adil, dan saya merasa bahwa saya tidak dianggap sebagai seorang yang bertanggung jawab dan tidak dianggap sebagai seorang yang bisa di percaya. Saya merasa bahwa saya dihakimi dan dihakimi oleh atasan saya, dan saya merasa bahwa saya tidak dianggap sebagai seorang yang beradab dan tidak dianggap sebagai seorang yang bisa di percaya.

Saya juga merasa bahwa hal ini tidak adil dan tidak adil, dan saya merasa bahwa saya tidak dianggap sebagai seorang yang bisa di percaya, dan saya


In [69]:
system_prompts_ID = {
    'anger': "Anda merasa frustrasi dan jengkel",
    'happiness': "Anda merasa bahagia dan optimis. Ungkapkan pikiran Anda dengan antusiasme dan kehangatan. Gunakan bahasa yang membangkitkan semangat. Tunjukkan apresiasi dan energi positif.",
    'sadness': "Anda merasa murung dan reflektif. Ungkapkan pikiran Anda dengan kelembutan dan introspeksi. Gunakan bahasa yang penuh perenungan. Tunjukkan kerentanan dan kedalaman emosi.",
    'fear': "Anda merasa cemas dan waspada. Ungkapkan pikiran Anda dengan keraguan dan kekhawatiran. Gunakan bahasa yang hati-hati dan tidak pasti. Tunjukkan kewaspadaan dan kegelisahan.",
    'love': "Anda merasa penuh kasih dan kepedulian. Ungkapkan pikiran Anda dengan kelembutan dan kedekatan. Gunakan bahasa yang hangat dan penuh afeksi. Tunjukkan empati dan pengertian."
}

In [82]:
text_indo_anger = generateSteering(
    user_text=prompt,
    system_text=system_prompt_ID + system_prompts_ID['anger'],
    model=model,
    steering_vector=None,
    tokenizer=tokenizer,
    steering_strength=0.3,
)

print(text_indo_anger)

Aku merasa frustrasi dan jengkel sekali. Aku tidak pernah merasa begitu sebelumnya. Aku telah bekerja keras dan selalu berusaha untuk melakukan yang terbaik dalam tim, tapi sekarang aku merasa seperti tidak ada gunanya. Aku tidak bisa percaya bahwa atasan aku bisa menyalahkan aku atas kesalahan yang bukan aku lakukan.

Aku ingat ketika rapat tim berlangsung, aku sedang fokus pada agenda yang lain dan tidak melihat kesalahan itu terjadi. Tapi aku tidak memiliki bukti apa pun untuk membuktikan bahwa aku tidak bersalah. Aku hanya dapat berharap bahwa atasan aku akan mendengarkan aku dan memahami situasinya.

Sekarang aku merasa seperti tidak ada gunanya berbicara atau berusaha keras dalam tim. Aku merasa seperti tidak ada yang percaya pada aku. Aku mer
